In [3]:
import os
import pandas as pd
import numpy as np

def main():
    # 1. Khai báo đường dẫn file CSV
    input_csv = "../outputs/evaluation_results_all_seeds.csv" 
    output_csv = "../outputs/final_aggregated_results.csv"
    
    if not os.path.exists(input_csv):
        input_csv = "outputs/evaluation_results_all_seeds.csv"
        output_csv = "outputs/final_aggregated_results.csv"
        if not os.path.exists(input_csv):
            print(f"[!] Lỗi: Không tìm thấy file {input_csv}")
            return

    print(f"[*] Đang đọc dữ liệu từ: {input_csv}")
    df = pd.read_csv(input_csv)
    print(f"[*] Tổng số bản ghi (rows) đã đọc: {len(df)}")

    # 2. Tách tên Cấu hình
    def get_config_name(ckpt_name):
        name = ckpt_name.replace("best_", "").split("_seed")[0]
        return name.upper()

    df['Config'] = df['Checkpoint'].apply(get_config_name)

    # 3. Các cột độ đo cần tính (Ép kiểu số để đảm bảo an toàn 100%)
    metrics = ['Macro_ROC_AUC', 'Macro_PR_AUC_mAP', 'Macro_F1_OptThresh']
    for col in metrics:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 4. Group by theo Config và tính Mean, Std
    grouped = df.groupby('Config')[metrics].agg(['mean', 'std'])

    # 5. Tạo DataFrame mới
    df_final = pd.DataFrame()
    df_final['Cấu hình (Loss)'] = grouped.index

    # Vòng lặp format chuỗi "Mean ± Std"
    for metric in metrics:
        # SỬA LỖI Ở ĐÂY: Thêm .values để bỏ qua việc căn chỉnh Index của Pandas
        df_final[metric] = grouped[metric].apply(
            lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}", axis=1
        ).values 

    # 6. Sắp xếp bảng cho đẹp
    df_final = df_final.sort_values(by='Cấu hình (Loss)').reset_index(drop=True)
    
    # Đổi tên cột
    df_final.rename(columns={
        'Macro_ROC_AUC': 'Macro ROC-AUC',
        'Macro_PR_AUC_mAP': 'Macro PR-AUC (mAP)',
        'Macro_F1_OptThresh': 'Macro F1 (Opt Thresh)'
    }, inplace=True)

    # 7. In kết quả ra màn hình
    print("\n" + "="*90)
    print(" BẢNG KẾT QUẢ TỔNG HỢP (MEAN ± STD) KHẢO SÁT HÀM MẤT MÁT TỪ 3 SEEDS")
    print("="*90 + "\n")
    print(df_final.to_markdown(index=False))

    # 8. Lưu ra file CSV mới
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df_final.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n[*] Đã lưu bảng tổng hợp ra file: {output_csv}")

if __name__ == "__main__":
    main()

[*] Đang đọc dữ liệu từ: ../outputs/evaluation_results_all_seeds.csv
[*] Tổng số bản ghi (rows) đã đọc: 33

 BẢNG KẾT QUẢ TỔNG HỢP (MEAN ± STD) KHẢO SÁT HÀM MẤT MÁT TỪ 3 SEEDS

| Cấu hình (Loss)   | Macro ROC-AUC   | Macro PR-AUC (mAP)   | Macro F1 (Opt Thresh)   |
|:------------------|:----------------|:---------------------|:------------------------|
| ASL_A1            | 0.8552 ± 0.0166 | 0.3747 ± 0.0153      | 0.4207 ± 0.0166         |
| ASL_A2            | 0.8604 ± 0.0064 | 0.3910 ± 0.0162      | 0.4351 ± 0.0176         |
| ASL_A3            | 0.8588 ± 0.0026 | 0.3709 ± 0.0029      | 0.4124 ± 0.0020         |
| BCE               | 0.8711 ± 0.0359 | 0.3866 ± 0.0336      | 0.4261 ± 0.0276         |
| FOCAL_F1          | 0.8819 ± 0.0102 | 0.3850 ± 0.0207      | 0.4189 ± 0.0178         |
| FOCAL_F2          | 0.8697 ± 0.0084 | 0.3924 ± 0.0032      | 0.4322 ± 0.0046         |
| FOCAL_F3          | 0.8724 ± 0.0160 | 0.3890 ± 0.0139      | 0.4264 ± 0.0095         |
| FOCAL_F4          | 